# ComfyUI + Pony Realism on Google Colab

Use a GPU runtime (**Runtime → Change runtime type → T4 GPU**). Run the cells from top to bottom. The model download is large (~6.6 GB).

In [ ]:
# Clone ComfyUI and install its Python requirements
from pathlib import Path
import subprocess, os

if not Path('/content/ComfyUI').exists():
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', '/content/ComfyUI'], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-r', '/content/ComfyUI/requirements.txt'], check=True)

# Download Pony Realism checkpoint only if it isn't already present
checkpoint = Path('/content/ComfyUI/models/checkpoints/ponyRealism.safetensors')
checkpoint.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint.exists() or checkpoint.stat().st_size < 1_000_000_000:
    url = 'https://huggingface.co/LyliaEngine/ponyRealism_v21MainVAE/resolve/main/ponyRealism_v21MainVAE.safetensors?download=true'
    subprocess.run(['wget', '-c', '-O', str(checkpoint), url], check=True)
print(f'Checkpoint ready: {checkpoint} ({checkpoint.stat().st_size / (1024**3):.2f} GiB)')

## Start ComfyUI and create a temporary link

Run this cell after the download finishes. Keep it running while using ComfyUI. Open the `https://….trycloudflare.com` URL printed by Cloudflare.

In [ ]:
import subprocess, time, os, urllib.request

# Start ComfyUI in the background
subprocess.run('pkill -f "python main.py --listen 127.0.0.1 --port 8188" || true', shell=True)
server_log = open('/content/comfyui.log', 'w')
server = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    cwd='/content/ComfyUI', stdout=server_log, stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait up to 3 minutes for ComfyUI to start
ready = False
for _ in range(90):
    if server.poll() is not None:
        raise RuntimeError('ComfyUI exited early. Check /content/comfyui.log for details.')
    try:
        urllib.request.urlopen('http://127.0.0.1:8188', timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(2)
if not ready:
    raise RuntimeError('ComfyUI did not become ready. Check /content/comfyui.log.')
print('ComfyUI is running. Starting Cloudflare Quick Tunnel…')

# Download Cloudflare tunnel client
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/content/cloudflared'], check=True)
os.chmod('/content/cloudflared', 0o755)

# Keep this cell running; Cloudflare prints the public temporary URL
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in tunnel.stdout:
    print(line, end='')

## First image

In ComfyUI, choose `ponyRealism.safetensors` in the checkpoint loader. Start with a simple text-to-image workflow and modest settings (for example, 768×1024 or 1024×768, 20–30 steps, batch size 1). If the UI cannot load, check the setup cell and `/content/comfyui.log`.

**Safety note:** the Quick Tunnel link is accessible to anyone who has it. Don't put secrets or private files in this session.